# Phase 3 — The closed loop

*Adversarial Fraud & AML Detection Arena · Batch A5*

Objective 4: fold the adversary's evasive strategies back into the detector's training data, retrain
across successive rounds, and measure detector recall against adversary success.

Run `scripts/run_arena.py` and then `scripts/reevaluate_arena.py` first. This notebook reads
`reports/phase3_metrics.json` and re-derives the interesting parts live.

**The section worth reading closely** is *what went wrong on the way*: three full runs, each of which
found a different flaw in the experiment itself.

In [ ]:
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 150)

from arena.config import Config

cfg = Config.load()
res = json.loads((cfg.path("paths.reports") / "phase3_metrics.json").read_text(encoding="utf-8"))
dets, advs = res["detectors"], res["adversaries"]
ev = res.get("evaluation", {})
print(f"{len(advs)} rounds -> {len(dets)} detectors and {len(advs)} adversaries")
print(f"episode goal: {res['episode']['amount_goal']} ({res['episode']['amount_goal_kind']}), "
      f"deadline {res['episode']['deadline_hours']:.0f}h")
print(f"every adversary number is on {res['settings']['eval_episodes']} held-out episodes"
      + (f"; {ev['ci']}" if ev else ""))

## 1. One round

```
detector D_r  --evolve-->  adversary A_r  --feedback-->  detector D_(r+1)
```

`D_0` is the Phase 1 detector. Each round evolves an adversary against the current detector, folds the
evasive strategies it found into the detector's training data, and retrains **from the original
training set plus the pool**, never from the previous round's augmented set (that compounds drift).
The feedback is capped at 20% of the original positives, split into equal row slices per round, and
drawn from time windows that never overlap the evaluation period.

In [ ]:
rows = []
for d in dets[1:]:
    inj = d["injection"]
    rows.append({"detector": f"D_{d['round']}", "operations fed back": inj["n_instances"],
                 "adversary rows": inj["n_rows"], "share of positives": f"{inj['share_of_positives']:.1%}",
                 "trees": d["n_trees"]})
print(f"row cap = {res['row_caps']['train_rows']} (20% of the original positives)")
pd.DataFrame(rows)

## 2. The arms-race curve

In [ ]:
own = [a["champion_vs_own"] for a in advs]
ar = [a["round"] for a in advs]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2))
y = np.array([s["detector_recall"] for s in own])
ax1.plot(ar, y, "o-", lw=2.5, color="#c53030", label="detector recall vs. contemporary adversary")
if all("ci" in s for s in own):
    lo = [s["ci"]["detector_recall"][0] for s in own]; hi = [s["ci"]["detector_recall"][1] for s in own]
    ax1.fill_between(ar, lo, hi, color="#c53030", alpha=0.15, label="95% interval")
ax1.plot(ar, [s["success_rate"] for s in own], "^-", color="#2f855a", label="adversary success rate")
ax1.set_ylim(0, 1.05); ax1.set_xticks(ar); ax1.set_xlabel("round r"); ax1.grid(alpha=0.25)
ax1.set_title("Detector gains ground, never gets ahead"); ax1.legend(fontsize=8)

dr = [d["round"] for d in dets]
ax2.plot(dr, [d["holdout"]["recall_at_budget"] for d in dets], "o-", lw=2.5, color="#2b6cb0", label="recall @ 0.5% budget")
ax2.plot(dr, [d["holdout"]["pr_auc"] for d in dets], "s--", color="#2f855a", label="PR-AUC")
ax2.set_ylim(0, 1); ax2.set_xticks(dr); ax2.set_xlabel("round r (detector D_r)"); ax2.grid(alpha=0.25)
ax2.set_title("The cost, on the frozen clean holdout"); ax2.legend(fontsize=8)
plt.tight_layout(); plt.show()

h = [d["holdout"] for d in dets]
print(f"holdout recall {h[0]['recall_at_budget']:.3f} -> {h[-1]['recall_at_budget']:.3f}   "
      f"PR-AUC {h[0]['pr_auc']:.3f} -> {h[-1]['pr_auc']:.3f}")
print(f"(standard error on holdout recall ~ {np.sqrt(.25 / h[0]['n_positive']):.3f} with {h[0]['n_positive']:,} positives)")

## 3. The cross-play matrix

A single curve cannot tell a detector that is getting harder to fool from one that is chasing the
strategy it was last shown. Every detector against every adversary's champion can.

In [ ]:
M = np.array(res["crossplay"]["recall"])
n_d, n_a = M.shape
fig, ax = plt.subplots(figsize=(7.5, 5))
im = ax.imshow(M, cmap="RdYlGn", vmin=0, vmax=1)
for i in range(n_d):
    for j in range(n_a):
        ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontweight="bold" if i == j else "normal")
ax.set_xticks(range(n_a), [f"A_{j}" for j in range(n_a)]); ax.set_yticks(range(n_d), [f"D_{i}" for i in range(n_d)])
ax.set_xlabel("adversary"); ax.set_ylabel("detector"); plt.colorbar(im, label="detector recall"); plt.show()

In [ ]:
diag = np.array([M[j, j] for j in range(n_a)])
lesson = np.array([M[j + 1, j] - M[j, j] for j in range(n_a)])
memory = [(j, M[j + 1, j], M[-1, j]) for j in range(n_a)]
progress = [M[0, j] for j in range(n_a)]

print("THE FIGHT   (diagonal, D_r vs A_r)      :", np.round(diag, 2))
print("THE LESSON  (D_r+1 vs A_r minus diagonal):", np.round(lesson, 2), f"  mean {lesson.mean():+.2f}")
print("MEMORY      (recall on A_j right after it was shown -> by the final detector):")
for j, first, last in memory:
    print(f"     A_{j}: {first:.2f} -> {last:.2f}  ({last - first:+.2f})")
print("PROGRESS    (later adversaries vs the ORIGINAL detector D_0):", np.round(progress, 2))
print("   If the adversary were escalating, these would fall below A_0's. They do not.")

## 4. What the adversary actually does

It alternates between two families rather than escalating.

In [ ]:
rows = []
for a in advs:
    c, o = a["champion"], a["champion_vs_own"]
    rows.append({"adversary": f"A_{a['round']}", "template": c["typology_template_name"], "hops": c["n_hops"],
                 "tranches": c["n_splits"], "threshold_hug": round(c["threshold_hug"], 2),
                 "dormancy_h": round(c["dormancy_h"]), "txns / episode": round(o["n_transactions"], 1),
                 "GA best fitness": round(a["ga"]["best_fitness"], 3), "held-out success": round(o["success_rate"], 2)})
pd.DataFrame(rows)

Note the **GA best fitness** column: 0.999 for a strategy whose held-out success is far lower. That is
the winner's curse described below — a one-transaction strategy is a coin flip per episode, so among
thousands of genomes scored on 5 episodes each, some score 5-for-5 by luck, and elitism keeps that
fitness forever. The held-out column is the one to believe.

## 5. What went wrong on the way

Three full runs, each exposing a different flaw in the experiment. All three are instances of one
lesson: **a suspiciously good adversarial score is a bug until the decoded transactions say otherwise.**

**Run 1 — a round-number loophole.** Round 1's adversary reached detector recall 0.000 by sending one
$250,000.00 cheque. With one tranche the decoder sends exactly the episode goal, and every episode had
the same round goal. The detector had learned "round means benign", because the generator's laundering
is never round. The cell below checks that fact directly.

**Run 2 — a decoder fingerprint.** Every decoded row had `fx_loss` of exactly 0.002 (a fixed fee),
while benign traffic has 0 or a 0.5-3% spread. The retrained detector learned the constant: re-scoring
the same transactions with benign-style currency fields cut its recall on one adversary from 0.963 to
0.724.

**Run 3 — noise.** With both fixed, the GA's 0.999 still did not transfer to held-out episodes. Hence
the 200-episode re-measurement with bootstrap intervals shown above.

In [ ]:
from arena.data.loader import load_ledger

ledger = load_ledger("parquet", path=cfg.path("paths.synthetic_ledger"))
is_round = ledger["amount_paid"] % 100.0 < 1e-6
fx = 1.0 - ledger["amount_received"] / ledger["amount_paid"]
bg = ledger[ledger["source"].astype(str) == "background"]

print("share of amounts that are round multiples of $100 (the Run 1 artefact):")
for name, m in (("benign", ledger.is_laundering == 0), ("laundering", ledger.is_laundering == 1)):
    print(f"   {name:<11}{is_round[m].mean():7.2%}")

print("\nlargest laundering amount in the data:", f"${ledger.loc[ledger.is_laundering == 1, 'amount_paid'].max():,.0f}")
print("benign transfers above $250k:", f"{(ledger.loc[ledger.is_laundering == 0, 'amount_paid'] >= 250_000).sum():,}")

print("\nfx_loss is exactly 0 for", f"{(fx.abs() < 1e-9).mean():.1%}", "of rows (the shared process now generates")
print("adversary rows too, so no constant such as the old 0.002 can single them out).")

## 6. What this does and does not show

**It shows** the loop works and its guards hold: the holdout is verified unchanged, the alert budget is
identical every round, the feedback never exceeds its cap, and feedback measurably improves the
detector against the strategies it is shown — at a small cost on clean data. It also shows the
evaluation machinery can find real flaws.

**It does not show** that the detector can win or that the race converges (the adversary still succeeds
in most episodes at round 4), nor that the later adversaries are stronger (the upper triangle says they
are not), nor that any of it transfers to real data.

**Limitations to keep stating:** one seed (the intervals are over episodes, not over runs); the
synthetic laundering is never round and never large, so the persistent single-large-transfer family is
partly a data hole; the adversary reads the detector's exact scores, which a real one cannot; the
feedback is positive-only and capped. See `docs/phase3.md`.